# Stage 1 — Epoch 5 evaluation with budget forcing
Default checkpoint: `TRAIN_ROOT/checkpoints/stage1/sft_s1k_deepseek_pro/epoch_5/`.
Evaluate **AIME 2024, AIME 2025, AIME 2026, AMC 2023, and MATH-500** with
**temperature 1.0, top-p 0.7, one answer per problem** (630 responses).

Set **MAX_THINKING_TOKENS** in the settings cell. Its default is **18,432**;
the answer allowance is automatically **20,480 − MAX_THINKING_TOKENS** (default **2,048**).
Minimum reasoning stays zero. At the cap, append `</think>`
and `Final Answer:` and continue the same assistant response. A natural `</think>`
starts the answer phase earlier; natural EOS is respected. No "Wait" extension is used.
The answer allowance includes injected delimiter/cue tokens, keeping the full continuation
within **20,480 tokens**. The first-phase count includes the generated opening think marker.

Each budget gets a separate folder, by default
`profiles/sample1_budget/thinking_18432_answer_2048/`. Changing the cap starts a separate
evaluation; rerun the checkpoint-check cell after editing settings. The resolved budget
YAML is saved to Drive under `TRAIN_ROOT/eval_configs/` and used by both smoke and full runs.
Raw records include whether the transition was forced and each phase's token counts.
Training, prompts, benchmarks, and answer scoring are unchanged. No baseline is required.

Use the **RTX PRO 6000 Blackwell 96GB**, enable `HF_TOKEN`, and push the updated Python
files/configs to GitHub main before setup. Run the optional five-problem smoke cell first,
then enable the separate full evaluation cell. Default Run all performs setup/checks only.
Epoch 5 is selected by default; earlier epochs remain optional.

To test the **previous R1 SFT checkpoint** with the same budget forcing, change CONFIG
below to `configs/stage1_sft.yaml`; this selects `sft_s1k/epoch_5` and a separate result name.
To run without budget forcing, select `sample1` and use a fresh BASELINE_ROOT if that
profile already has manifests from an older evaluator version. Existing runs require
their original code/settings to resume; old generations are never imported here.

In [ ]:
REPO_URL = "https://github.com/seungjun-green/post-training-aime.git"
CODE_ROOT = "/content/lg-aime-stage1-eval"
TRAIN_ROOT = "/content/drive/MyDrive/LG-AIME-Stage1"
BASELINE_ROOT = "/content/drive/MyDrive/LG-AIME-English-Eval-compatible"
EVAL_ENV = "/content/lg-eval-env"
CONFIG = "configs/stage1_sft_deepseek_pro.yaml"
EVAL_PROFILE = "sample1_budget" # @param ["sample1_budget", "sample1"]
MAX_THINKING_TOKENS = 18432 # @param {type:"integer"}

In [ ]:
import os, subprocess, sys, json
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
if not Path(CODE_ROOT).exists():
    subprocess.check_call(["git", "clone", "--branch", "main", REPO_URL, CODE_ROOT])
def git(*args):
    return subprocess.check_output(["git", "-C", CODE_ROOT, *args], text=True).strip()
if git("remote", "get-url", "origin") != REPO_URL or git("status", "--porcelain"):
    raise ValueError("Use a clean checkout of the configured repository")
if git("branch", "--show-current") != "main":
    raise ValueError("Use a main checkout")
subprocess.check_call(["git", "-C", CODE_ROOT, "pull", "--ff-only", "origin", "main"])
print("Evaluation code commit:", git("rev-parse", "HEAD"))
if (not (Path(CODE_ROOT) / CONFIG).is_file()
        or not (Path(CODE_ROOT) / "eval/budget_generation.py").is_file()):
    raise FileNotFoundError(f"Missing {CONFIG} or budget-forcing code. Push all evaluation updates to GitHub main, then rerun setup.")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv==0.11.22"])
subprocess.check_call([sys.executable, "scripts/setup_eval_runtime.py", "--venv", EVAL_ENV], cwd=CODE_ROOT)
sys.path.insert(0, CODE_ROOT)
from common.process import run_logged

## Check saved checkpoints
Keep `TRAIN_ROOT` the same as in the training notebook. SFT evaluation can run before
the baseline. The first run creates the selected profile's protocol/runtime manifests
under `BASELINE_ROOT/profiles/<profile>/full/results/`. Later runs validate the same
sampling, prompts, datasets, scoring and runtime for comparisons within that profile.
To switch from an older notebook, stop evaluation, push the updated code, and run setup.
New profiles start fresh and leave previous 32-sample results intact; old responses are
not imported. Once a run starts, keep its code commit and settings unchanged for resume.

In [ ]:
import yaml
from eval.profiles import profile_root, load_profile
eval_config = yaml.safe_load((Path(CODE_ROOT) / 'configs/eval_english.yaml').read_text())
execution = yaml.safe_load((Path(CODE_ROOT) / 'configs/eval_execution.yaml').read_text())
budget = None
BUDGET_ARGS = []
EVAL_LABEL = EVAL_PROFILE
if EVAL_PROFILE == 'sample1_budget':
    if type(MAX_THINKING_TOKENS) is not int or not 0 < MAX_THINKING_TOKENS < eval_config['max_new_tokens']:
        raise ValueError('MAX_THINKING_TOKENS must be a positive integer below the total token budget')
    budget = yaml.safe_load((Path(CODE_ROOT) / 'configs/eval_profiles.yaml').read_text())[EVAL_PROFILE]['budget_forcing']
    budget['max_reasoning_tokens'] = MAX_THINKING_TOKENS
    budget['answer_budget_tokens'] = eval_config['max_new_tokens'] - MAX_THINKING_TOKENS
    load_profile(EVAL_PROFILE, eval_config, execution, budget_override=budget)
    EVAL_LABEL += f"_thinking_{MAX_THINKING_TOKENS}_answer_{budget['answer_budget_tokens']}"
    budget_path = Path(TRAIN_ROOT) / 'eval_configs' / (EVAL_LABEL + '.yaml')
    budget_path.parent.mkdir(parents=True, exist_ok=True)
    budget_path.write_text(yaml.safe_dump(budget, sort_keys=False))
    BUDGET_ARGS = ['--budget_config', str(budget_path)]
    print('Budget config saved:', budget_path)
INCLUDE_EARLIER_EPOCHS = False
cfg = yaml.safe_load((Path(CODE_ROOT) / CONFIG).read_text())
epochs = int(cfg["training"]["num_train_epochs"])
selected_epochs = [epochs]
if INCLUDE_EARLIER_EPOCHS:
    selected_epochs.extend(range(1, epochs))
checkpoint_root = Path(TRAIN_ROOT) / "checkpoints" / cfg["stage"] / cfg["run_name"]
result_root = profile_root(BASELINE_ROOT, EVAL_PROFILE, budget) / "full/results"
run_identities = set()
eval_runs = []
for epoch in selected_epochs:
    checkpoint = checkpoint_root / f"epoch_{epoch}"
    marker = json.loads((checkpoint / "stage1_checkpoint.json").read_text())
    if marker["epoch"] != epoch:
        raise ValueError(f"Checkpoint epoch mismatch: {checkpoint}")
    run_identities.add(marker["run_identity"])
    # Profiles use separate directories for baseline and SFT comparisons.
    run_name = cfg["run_name"] if epoch == epochs else f"{cfg['run_name']}_epoch{epoch}"
    eval_runs.append((epoch, checkpoint, run_name))
if len(run_identities) != 1:
    raise ValueError("Selected epoch checkpoints must belong to the same training run")
suite = json.loads((Path(CODE_ROOT) / "configs/english_eval_suite.json").read_text())
progress_totals = {name: entry["rows"] for name, entry in suite["datasets"].items()}
print("Benchmarks:", ", ".join(progress_totals))
for epoch, checkpoint, run_name in eval_runs:
    print(f"Epoch {epoch}: {checkpoint} -> {result_root / cfg['stage'] / (run_name + '.json')}")
print("Evaluation profile:", EVAL_PROFILE)
print("Budget settings:", budget or "off")

## Optional smoke test — five problems, epoch 5
Enable this to test one problem from each benchmark before the full run. It uses the same
checkpoint, sampling, and budget settings. Smoke results and raw generations are isolated
under `profiles/<profile>/smoke/`; they are not counted as full evaluation results.
Check `budget_forcing` in the generations JSONL for phase lengths and `forced` transitions.
A short smoke response may finish naturally without exercising the reasoning cap.

In [ ]:
RUN_SMOKE_EVAL = False
if RUN_SMOKE_EVAL:
    epoch, checkpoint, run_name = eval_runs[0]
    command = [str(Path(EVAL_ENV) / "bin/python"), "-m", "eval.run_batched_eval",
               "--model", str(checkpoint), "--stage", cfg["stage"], "--run_name", run_name,
               "--output_root", BASELINE_ROOT, "--profile", EVAL_PROFILE,
               "--execution_config", "configs/eval_execution.yaml", "--smoke"]
    command.extend(BUDGET_ARGS)
    run_logged(command, cwd=CODE_ROOT,
               log_path=Path(TRAIN_ROOT) / f"{run_name}_{EVAL_LABEL}_smoke_console.log",
               progress_totals={name: 1 for name in progress_totals})
else:
    print("Smoke evaluation is off. Enable RUN_SMOKE_EVAL to test five problems.")

## Full evaluation — epoch 5, then optional earlier checkpoints
Enable `RUN_STAGE1_EVAL` to evaluate epoch 5. Leave `INCLUDE_EARLIER_EPOCHS = False`
above for the main result only. To include the earlier checkpoints, set it to `True`
and rerun the check cell; the evaluation order will be **5, 1, 2, 3, 4**.
A default Run all performs setup and checks only. Every selected epoch uses all five benchmarks, with the same
prompts, sampling, seeds and scoring as the baseline. Epoch 5 remains the Stage 1 result.
Reports, raw generations and manifests are saved under `BASELINE_ROOT/profiles/<profile>/full/results/stage1/`:
`sft_s1k_deepseek_pro_epoch1.json` through `sft_s1k_deepseek_pro_epoch4.json`, and `sft_s1k_deepseek_pro.json` for epoch 5. Completed problems are durably saved during the run
in `*_problems.jsonl`; the standard `*_generations.jsonl` export is written at completion.
Console logs are saved under `TRAIN_ROOT`. Repeating the same commands resumes completed
problems. `configs/eval_profiles.yaml` defines the selected profile: temperature 1.0, top-p 0.7,
one answer per problem, and up to 64 pending problems for 32 GPU response slots. Exact sampled text can
change with batching even though seeds and sampling settings are unchanged.
Progress counts completed problems, which can finish out of order within each benchmark.
No AMC-only runs are launched by this notebook.

In [ ]:
RUN_STAGE1_EVAL = False
if RUN_STAGE1_EVAL:
    eval_python = str(Path(EVAL_ENV) / "bin/python")
    for epoch, checkpoint, run_name in eval_runs:
        command = [eval_python, "-m", "eval.run_batched_eval", "--model", str(checkpoint),
                   "--stage", cfg["stage"], "--run_name", run_name, "--output_root", BASELINE_ROOT,
                   "--profile", EVAL_PROFILE,
                   "--execution_config", "configs/eval_execution.yaml"]
        command.extend(BUDGET_ARGS)
        run_logged(command,
                   cwd=CODE_ROOT, log_path=Path(TRAIN_ROOT) / f"{run_name}_epoch{epoch}_{EVAL_LABEL}_console.log",
                   progress_totals=progress_totals)
else:
    print("Evaluation is off. Enable RUN_STAGE1_EVAL to run the selected checkpoints, starting with epoch 5.")